### MultiModel RAG

In [1]:
import pymupdf
from langchain_core.documents import Document
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

d:\Agentic-LangGraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\venka\AppData\Local\Temp\ipykernel_28396\1303238037.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 22363.33it/s]


CLIPModel(
  (text_model): CLIPTextModel(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1

In [3]:
import torch
import torch.nn.functional as F
from PIL import Image

# Image Embedding
def embed_img(image_data):
    """Embed image using CLIP"""

    if isinstance(image_data, str):
        image = Image.open(image_data).convert("RGB")
    else:
        image = image_data.convert("RGB")

    inputs = clip_processor(images=image, return_tensors="pt")

    with torch.no_grad():
        features = clip_model.get_image_features(**inputs)

        # Extract tensor if an output object is returned
        if not isinstance(features, torch.Tensor):
            features = features.pooler_output

        # Normalize embedding
        features = F.normalize(features, p=2, dim=-1)

        return features.squeeze().cpu().numpy()


# Text Embedding
def embed_text(text):
    """Embed text using CLIP"""

    inputs = clip_processor(
        text=[text],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77
    )

    with torch.no_grad():
        features = clip_model.get_text_features(**inputs)

        # Extract tensor if an output object is returned
        if not isinstance(features, torch.Tensor):
            features = features.pooler_output

        # Normalize embedding
        features = F.normalize(features, p=2, dim=-1)

        return features.squeeze().cpu().numpy()


In [5]:
## Process PDF
pdf_path=r"D:\Agentic-LangGraph\multimodal_sample.pdf"
doc=pymupdf.open(pdf_path)
# Storage for all documents and embeddings
all_docs = []
all_embeddings = []
image_data_store = {}  # Store actual image data for LLM

# Text splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)




In [6]:
import pymupdf

pdf_path = r"D:\Agentic-LangGraph\multimodal_sample.pdf"

doc = pymupdf.open(pdf_path)

print("PDF opened:", not doc.is_closed)
print("Total pages:", doc.page_count)

PDF opened: True
Total pages: 1


In [7]:

# Reset the lists before processing
all_docs = []
all_embeddings = []
image_data_store = {}

for i, page in enumerate(doc):

    # Extract text from the current page
    text = page.get_text()

    if text.strip():
        temp_doc = Document(
            page_content=text,
            metadata={"page": i, "type": "text"}
        )

        text_chunks = splitter.split_documents([temp_doc])

        for chunk in text_chunks:
            embedding = embed_text(chunk.page_content)

            all_embeddings.append(embedding)
            all_docs.append(chunk)

    # Extract images from the current page
    for img_index, img in enumerate(page.get_images(full=True)):
        try:
            xref = img[0]

            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]

            pil_image = Image.open(
                io.BytesIO(image_bytes)
            ).convert("RGB")

            image_id = f"page_{i}_img_{img_index}"

            # Convert image to Base64
            buffered = io.BytesIO()
            pil_image.save(buffered, format="PNG")

            img_base64 = base64.b64encode(
                buffered.getvalue()
            ).decode()

            # Generate image embedding
            embedding = embed_img(pil_image)

            all_embeddings.append(embedding)

            # Store the image
            image_data_store[image_id] = img_base64

            # Create document metadata
            image_doc = Document(
                page_content=f"[Image: {image_id}]",
                metadata={
                    "page": i,
                    "type": "image",
                    "image_id": image_id
                }
            )

            all_docs.append(image_doc)

        except Exception as e:
            print(
                f"Error processing image {img_index} "
                f"on page {i}: {e}"
            )

print("PDF processing completed!")
print("Total documents:", len(all_docs))
print("Total embeddings:", len(all_embeddings))


PDF processing completed!
Total documents: 2
Total embeddings: 2


In [8]:
all_docs

[Document(metadata={'page': 0, 'type': 'text'}, page_content='Annual Revenue Overview\nThis document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart\nbelow, revenue grew steadily with the highest growth recorded in Q3.\nQ1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed\nQ1 due to marketing campaigns. Q3 had exponential growth due to global expansion.'),
 Document(metadata={'page': 0, 'type': 'image', 'image_id': 'page_0_img_0'}, page_content='[Image: page_0_img_0]')]

In [9]:
# Create unified FAISS vector store with CLIP embeddings
embeddings_array = np.array(all_embeddings)
embeddings_array

array([[-0.00267244,  0.01283002, -0.05183142, ..., -0.00385083,
         0.02977718, -0.00010684],
       [ 0.01732334, -0.01327685, -0.0242703 , ...,  0.08994089,
        -0.00272146,  0.03253034]], shape=(2, 512), dtype=float32)

In [10]:
(all_docs,embeddings_array)

([Document(metadata={'page': 0, 'type': 'text'}, page_content='Annual Revenue Overview\nThis document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart\nbelow, revenue grew steadily with the highest growth recorded in Q3.\nQ1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed\nQ1 due to marketing campaigns. Q3 had exponential growth due to global expansion.'),
  Document(metadata={'page': 0, 'type': 'image', 'image_id': 'page_0_img_0'}, page_content='[Image: page_0_img_0]')],
 array([[-0.00267244,  0.01283002, -0.05183142, ..., -0.00385083,
          0.02977718, -0.00010684],
        [ 0.01732334, -0.01327685, -0.0242703 , ...,  0.08994089,
         -0.00272146,  0.03253034]], shape=(2, 512), dtype=float32))

In [11]:
vector_store = FAISS.from_embeddings(
    text_embeddings=[(doc.page_content, emb) for doc, emb in zip(all_docs, embeddings_array)],
    embedding=None,  # We're using precomputed embeddings
    metadatas=[doc.metadata for doc in all_docs]
)
vector_store

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [12]:
# Initialize GPT-4 Vision model
llm = init_chat_model("openai:gpt-4.1")
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.9', 'langchain': '1.4.4', 'langchain-openai': '1.7.0'}}, profile={'name': 'GPT-4.1', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002365852C2F0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002365852CD70>, root_client=<openai.OpenAI object at 0x0000023656A2F8C0>, r

In [13]:

def retrieve_multimodal(query, k=5):
    """Unified retrieval using CLIP embeddings for both text and images."""
    # Embed query using CLIP
    query_embedding = embed_text(query)
    
    # Search in unified vector store
    results = vector_store.similarity_search_by_vector(
        embedding=query_embedding,
        k=k
    )
    
    return results

In [14]:
def create_multimodal_message(query, retrieved_docs):
    """Create a message with both text and images for GPT-4V."""
    content = []
    
    # Add the query
    content.append({
        "type": "text",
        "text": f"Question: {query}\n\nContext:\n"
    })
    
    # Separate text and image documents
    text_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "text"]
    image_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "image"]
    
    # Add text context
    if text_docs:
        text_context = "\n\n".join([
            f"[Page {doc.metadata['page']}]: {doc.page_content}"
            for doc in text_docs
        ])
        content.append({
            "type": "text",
            "text": f"Text excerpts:\n{text_context}\n"
        })
    
    # Add images
    for doc in image_docs:
        image_id = doc.metadata.get("image_id")
        if image_id and image_id in image_data_store:
            content.append({
                "type": "text",
                "text": f"\n[Image from page {doc.metadata['page']}]:\n"
            })
            content.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{image_data_store[image_id]}"
                }
            })
    
    # Add instruction
    content.append({
        "type": "text",
        "text": "\n\nPlease answer the question based on the provided text and images."
    })
    
    return HumanMessage(content=content)

In [15]:
def multimodal_pdf_rag_pipeline(query):
    """Main pipeline for multimodal RAG."""
    # Retrieve relevant documents
    context_docs = retrieve_multimodal(query, k=5)
    
    # Create multimodal message
    message = create_multimodal_message(query, context_docs)
    
    # Get response from GPT-4V
    response = llm.invoke([message])
    
    # Print retrieved context info
    print(f"\nRetrieved {len(context_docs)} documents:")
    for doc in context_docs:
        doc_type = doc.metadata.get("type", "unknown")
        page = doc.metadata.get("page", "?")
        if doc_type == "text":
            preview = doc.page_content[:100] + "..." if len(doc.page_content) > 100 else doc.page_content
            print(f"  - Text from page {page}: {preview}")
        else:
            print(f"  - Image from page {page}")
    print("\n")
    
    return response.content

In [16]:
if __name__ == "__main__":
    # Example queries
    queries = [
        "What does the chart on page 1 show about revenue trends?",
        "Summarize the main findings from the document",
        "What visual elements are present in the document?"
    ]
    
    for query in queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        answer = multimodal_pdf_rag_pipeline(query)
        print(f"Answer: {answer}")
        print("=" * 70)


Query: What does the chart on page 1 show about revenue trends?
--------------------------------------------------

Retrieved 2 documents:
  - Text from page 0: Annual Revenue Overview
This document summarizes the revenue trends across Q1, Q2, and Q3. As illust...
  - Image from page 0


Answer: The chart on page 1 shows that revenue increased each quarter. Specifically, revenue started at a moderate level in Q1 (represented by the blue bar), rose significantly in Q2 (green bar), and then experienced the highest and steepest growth in Q3 (red bar). This indicates a steady upward trend, with growth accelerating the most in Q3, aligning with the text's explanation of exponential growth due to global expansion.

Query: Summarize the main findings from the document
--------------------------------------------------

Retrieved 2 documents:
  - Text from page 0: Annual Revenue Overview
This document summarizes the revenue trends across Q1, Q2, and Q3. As illust...
  - Image from page 0


An